# Lower-back acceleration versus acceleration + gyroscope: three-source transport benchmark

This controlled GPU benchmark asks one question: when the model is trained on two sources and evaluated on the third unseen source, does the lower-back gyroscope channel add reliable transport value beyond lower-back acceleration?

Safeguards: source is entirely held out; participants never span train and test; normalisation is fit on training windows only; no frozen cohort is accessed; and both representations use identical splits, seeds, network, optimiser, epochs, and source/class-balanced sampling.

In [1]:
from pathlib import Path
import gc
import sys

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import balanced_accuracy_score, brier_score_loss, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from models.stroke_gait_inception import InceptionBlock

P = ROOT / 'data/processed'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'This benchmark is intentionally configured to use the available GPU.'
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0))
SEEDS = [42, 137, 202]
EPOCHS = 8

Device: cuda | NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
# Felius + Voisard: map native 18-axis windows to [LB acceleration magnitude, LB gyroscope magnitude].
fv_raw = np.load(P / 'validated_gait_windows_float32.npy', mmap_mode='r')
fv_meta = pd.read_csv(P / 'validated_window_metadata.csv')
fv_keep = fv_meta.label.isin(['healthy', 'stroke']).to_numpy()
fv_meta = fv_meta.loc[fv_keep].reset_index(drop=True)
fv_raw = fv_raw[fv_keep]
fv_x = np.stack([np.linalg.norm(fv_raw[:, :, :3], axis=2), np.linalg.norm(fv_raw[:, :, 3:6], axis=2)], axis=2).astype('float32')

# Sint: use only the versioned adapter from Notebook 27.
sint_x = np.load(P / 'sint_lower_back_accel_gyro_windows_v1_float32.npy', mmap_mode='r')
sint_meta = pd.read_csv(P / 'sint_lower_back_accel_gyro_window_metadata_v1.csv')
assert sint_x.shape[0] == len(sint_meta) and sint_x.shape[1:] == (500, 2)

X = np.concatenate([fv_x, sint_x]).astype('float32')
meta = pd.concat([fv_meta, sint_meta], ignore_index=True)
meta = meta[meta.label.isin(['healthy', 'stroke'])].reset_index(drop=True)
meta['y'] = meta.label.eq('stroke').astype(int)
meta['source'] = meta.dataset_id.astype(str)
meta['group'] = meta.participant_key.astype(str)
assert len(X) == len(meta) and np.isfinite(X).all()
display(meta.groupby(['source', 'label']).agg(participants=('group', 'nunique'), windows=('group', 'size')))
print('Combined input:', X.shape, '| participants:', meta.group.nunique())

participants  windows
source               label                         
felius_2024          healthy            34     2921
                     stroke            129    13445
sint_maartenskliniek healthy            20     3069
                     stroke             10      984
voisard_2025         healthy            72     1039
                     stroke             49     1106

Combined input: (22564, 500, 2) | participants: 314


In [3]:
class Net(torch.nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.features = torch.nn.Sequential(
            InceptionBlock(channels), torch.nn.MaxPool1d(2), InceptionBlock(64), torch.nn.AdaptiveAvgPool1d(1)
        )
        self.classifier = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Dropout(0.3), torch.nn.Linear(64, 1))
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

def source_class_weights(frame):
    key = frame.source.astype(str) + '|' + frame.y.astype(str)
    counts = key.value_counts()
    return torch.tensor(key.map(lambda value: 1.0 / counts[value]).to_numpy(), dtype=torch.double)

def participant_metrics(model, array, frame, mean, std):
    z = torch.from_numpy(((array - mean) / std).transpose(0, 2, 1).astype('float32')).to(DEVICE)
    with torch.inference_mode():
        probabilities = torch.sigmoid(model(z)).cpu().numpy()
    person = frame.assign(probability=probabilities).groupby(['group', 'y'], as_index=False).probability.mean()
    return {
        'participants': len(person), 'healthy': int((person.y == 0).sum()), 'stroke': int((person.y == 1).sum()),
        'auroc': roc_auc_score(person.y, person.probability),
        'balanced_accuracy': balanced_accuracy_score(person.y, person.probability >= 0.5),
        'healthy_specificity': float((person.loc[person.y == 0, 'probability'] < 0.5).mean()),
        'brier': brier_score_loss(person.y, person.probability),
    }

def fit_and_score(train_mask, test_mask, channels, seed, data=X):
    torch.manual_seed(seed)
    train_x = data[train_mask][:, :, channels]
    # Fold-local normalisation: nothing from the held-out source participates here.
    mean = train_x.reshape(-1, len(channels)).mean(axis=0)
    std = train_x.reshape(-1, len(channels)).std(axis=0).clip(1e-4)
    z = torch.from_numpy(((train_x - mean) / std).transpose(0, 2, 1).astype('float32'))
    y = torch.from_numpy(meta.loc[train_mask, 'y'].to_numpy('float32'))
    sampler = WeightedRandomSampler(source_class_weights(meta.loc[train_mask]), len(z), replacement=True,
                                    generator=torch.Generator().manual_seed(seed + 1000))
    loader = DataLoader(TensorDataset(z, y), batch_size=128, sampler=sampler)
    model = Net(len(channels)).to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    for _ in range(EPOCHS):
        model.train()
        for batch_x, batch_y in loader:
            optimiser.zero_grad(set_to_none=True)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(model(batch_x.to(DEVICE)), batch_y.to(DEVICE))
            loss.backward()
            optimiser.step()
    model.eval()
    outcome = participant_metrics(model, data[test_mask][:, :, channels], meta.loc[test_mask].copy(), mean, std)
    del model, optimiser, loader, z, y
    gc.collect(); torch.cuda.empty_cache()
    return outcome

In [4]:
rows = []
for held_out_source in sorted(meta.source.unique()):
    test_mask = meta.source.eq(held_out_source).to_numpy()
    train_mask = ~test_mask
    assert not set(meta.loc[train_mask, 'group']).intersection(set(meta.loc[test_mask, 'group']))
    for seed in SEEDS:
        for representation, channels in [('lb_acceleration_only', [0]), ('lb_acceleration_plus_gyroscope', [0, 1])]:
            result = fit_and_score(train_mask, test_mask, channels, seed)
            rows.append({'held_out_source': held_out_source, 'seed': seed, 'representation': representation, **result})
            print('complete', held_out_source, seed, representation)

results = pd.DataFrame(rows)
results.to_csv(P / 'lower_back_6dof_three_source_transport_benchmark.csv', index=False)
summary = results.groupby(['held_out_source', 'representation'])[['auroc', 'balanced_accuracy', 'healthy_specificity', 'brier']].agg(['mean', 'std']).round(4)
display(summary)

complete felius_2024 42 lb_acceleration_only


complete felius_2024 42 lb_acceleration_plus_gyroscope


complete felius_2024 137 lb_acceleration_only


complete felius_2024 137 lb_acceleration_plus_gyroscope


complete felius_2024 202 lb_acceleration_only


...

complete voisard_2025 137 lb_acceleration_only


complete voisard_2025 137 lb_acceleration_plus_gyroscope


complete voisard_2025 202 lb_acceleration_only


complete voisard_2025 202 lb_acceleration_plus_gyroscope


auroc          \
                                                       mean     std   
held_out_source      representation                                   
felius_2024          lb_acceleration_only            0.8499  0.0077   
                     lb_acceleration_plus_gyroscope  0.8590  0.0066   
sint_maartenskliniek lb_acceleration_only            0.8700  0.0397   
                     lb_acceleration_plus_gyroscope  0.7717  0.0794   
voisard_2025         lb_acceleration_only            0.8921  0.0245   
                     lb_acceleration_plus_gyroscope  0.8640  0.0292   

                                                    balanced_accuracy          \
                                                                 mean     std   
held_out_source      representation                                             
felius_2024          lb_acceleration_only                      0.7658  0.0171   
                     lb_acceleration_plus_gyroscope            0.7151  0.1260   
sint_maartenskliniek lb_acceleration_only                      0.8083  0.0144   
                     lb_acceleration_plus_gyroscope            0.6833  0.0144   
voisard_2025         lb_acceleration_only                      0.7920  0.0427   
                     lb_acceleration_plus_gyroscope            0.7825  0.0221   

                                                    healthy_specificity  \
                                                                   mean   
held_out_source      representation                                       
felius_2024          lb_acceleration_only                        0.7745   
                     lb_acceleration_plus_gyroscope              0.8824   
sint_maartenskliniek lb_acceleration_only                        0.7500   
                     lb_acceleration_plus_gyroscope              0.4333   
voisard_2025         lb_acceleration_only                        0.6111   
                     lb_acceleration_plus_gyroscope              0.6806   

                                                              brier          
                                                        std    mean     std  
held_out_source      representation                                          
felius_2024          lb_acceleration_only            0.0679  0.1778  0.0596  
                     lb_acceleration_plus_gyroscope  0.1176  0.2696  0.1679  
sint_maartenskliniek lb_acceleration_only            0.0866  0.1574  0.0333  
                     lb_acceleration_plus_gyroscope  0.0764  0.2873  0.0733  
voisard_2025         lb_acceleration_only            0.1325  0.1797  0.0427  
                     lb_acceleration_plus_gyroscope  0.0773  0.1740  0.0134

In [5]:
acc = results.loc[results.representation.eq('lb_acceleration_only')].set_index(['held_out_source', 'seed'])
six = results.loc[results.representation.eq('lb_acceleration_plus_gyroscope')].set_index(['held_out_source', 'seed'])
delta = six[['auroc', 'balanced_accuracy', 'healthy_specificity', 'brier']].subtract(acc[['auroc', 'balanced_accuracy', 'healthy_specificity', 'brier']])
delta = delta.reset_index()
delta.to_csv(P / 'lower_back_6dof_three_source_transport_paired_deltas.csv', index=False)
display(delta.groupby('held_out_source')[['auroc', 'balanced_accuracy', 'healthy_specificity', 'brier']].agg(['mean', 'std']).round(4))
print('Interpretation constraint: this is a 3-source transport benchmark, not a clinical validation or a basis for touching frozen cohorts.')

auroc         balanced_accuracy          \
                        mean     std              mean     std   
held_out_source                                                  
felius_2024           0.0091  0.0043           -0.0507  0.1114   
sint_maartenskliniek -0.0983  0.0751           -0.1250  0.0250   
voisard_2025         -0.0281  0.0055           -0.0095  0.0498   

                     healthy_specificity           brier          
                                    mean     std    mean     std  
held_out_source                                                   
felius_2024                       0.1078  0.0679  0.0918  0.1083  
sint_maartenskliniek             -0.3167  0.1041  0.1299  0.0475  
voisard_2025                      0.0694  0.2074 -0.0057  0.0525

Interpretation constraint: this is a 3-source transport benchmark, not a clinical validation or a basis for touching frozen cohorts.


In [6]:
# The published Sint pipeline describes low-pass filtering (17 Hz acceleration, 15 Hz gyroscope).
# Apply this deterministic, per-window preprocessing identically to every source, before fold-local normalisation.
from scipy.signal import butter, sosfiltfilt
filtered = X.copy()
filtered[:, :, 0] = sosfiltfilt(butter(2, 17, btype='lowpass', fs=100, output='sos'), filtered[:, :, 0], axis=1)
filtered[:, :, 1] = sosfiltfilt(butter(2, 15, btype='lowpass', fs=100, output='sos'), filtered[:, :, 1], axis=1)
filter_rows = []
for held_out_source in sorted(meta.source.unique()):
    test_mask = meta.source.eq(held_out_source).to_numpy()
    for seed in SEEDS:
        result = fit_and_score(~test_mask, test_mask, [0, 1], seed, data=filtered)
        filter_rows.append({'held_out_source': held_out_source, 'seed': seed,
                            'representation': 'lb_acceleration_plus_gyroscope_lowpass_17hz_15hz', **result})
        print('complete filtered', held_out_source, seed)
filtered_results = pd.DataFrame(filter_rows)
filtered_results.to_csv(P / 'lower_back_6dof_lowpass_three_source_transport_benchmark.csv', index=False)
display(filtered_results.groupby('held_out_source')[['auroc', 'balanced_accuracy', 'healthy_specificity', 'brier']].agg(['mean', 'std']).round(4))
print('This test diagnoses a documented preprocessing mismatch; it does not tune on any frozen cohort.')

complete filtered felius_2024 42


complete filtered felius_2024 137


complete filtered felius_2024 202


complete filtered sint_maartenskliniek 42


complete filtered sint_maartenskliniek 137


...

complete filtered voisard_2025 42


complete filtered voisard_2025 137


complete filtered voisard_2025 202


auroc         balanced_accuracy          \
                        mean     std              mean     std   
held_out_source                                                  
felius_2024           0.8383  0.0190            0.6872  0.1348   
sint_maartenskliniek  0.7633  0.0679            0.7000  0.0433   
voisard_2025          0.8881  0.0034            0.7882  0.0070   

                     healthy_specificity           brier          
                                    mean     std    mean     std  
held_out_source                                                   
felius_2024                       0.8627  0.0945  0.2928  0.1736  
sint_maartenskliniek              0.5333  0.1258  0.2568  0.0670  
voisard_2025                      0.6852  0.0814  0.1599  0.0214

This test diagnoses a documented preprocessing mismatch; it does not tune on any frozen cohort.


## Decision use

Use the paired results as evidence about representation robustness. Retain both lower-back views when their source-wise trade-off differs; select neither as a final clinical model from this benchmark alone. Any later architecture refinement must keep the same leakage controls and compare against this recorded reference.